# Steam Gaming Trends — 01: Data Profiling

**Dataset**: Kaggle `fronkongames/steam-games-dataset` (CC0 Public Domain)  
Run `python data/download_data.py` first to populate `data/`.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')

DATA = Path('data')
FIG  = Path('outputs/figures')
FIG.mkdir(parents=True, exist_ok=True)

for f in [DATA / 'games.csv', DATA / 'recommendations.csv']:
    if not f.exists():
        raise FileNotFoundError(f'{f} not found — run data/download_data.py')

## 1. Load games.csv

In [ ]:
games = pd.read_csv(DATA / 'games.csv', low_memory=False)
print(f'games.csv: {len(games):,} rows × {games.shape[1]} columns')
games.dtypes

In [ ]:
games.head(3)

## 2. Null Audit — games.csv

In [ ]:
null_pct = games.isnull().mean().sort_values(ascending=False).mul(100).round(1)
null_pct[null_pct > 0]

## 3. Distributions of Key Numeric Columns

In [ ]:
num_cols = ['price', 'positive', 'negative', 'median_playtime_forever', 'peak_ccu']
present = [c for c in num_cols if c in games.columns]

fig, axes = plt.subplots(1, len(present), figsize=(4 * len(present), 4))
for ax, col in zip(axes, present):
    data = games[col].dropna()
    # log-scale for heavy right tails
    data_log = np.log1p(data)
    ax.hist(data_log, bins=50, color='steelblue', edgecolor='none')
    ax.set_title(f'log1p({col})')
    ax.set_xlabel('log1p value')
fig.suptitle('games.csv — Key Numeric Distributions (log-scale)')
fig.tight_layout()
fig.savefig(FIG / 'prof01_games_distributions.png', dpi=150)
plt.show()

## 4. Load recommendations.csv (sample)

The full recommendations file is ~41 M rows. We load a 500k sample for profiling — the full file is used in notebook 03.

In [ ]:
recs_sample = pd.read_csv(DATA / 'recommendations.csv', nrows=500_000, low_memory=False)
print(f'recommendations.csv sample: {len(recs_sample):,} rows × {recs_sample.shape[1]} columns')
recs_sample.dtypes

In [ ]:
recs_sample.head(3)

## 5. Null Audit — recommendations sample

In [ ]:
recs_sample.isnull().mean().sort_values(ascending=False).mul(100).round(1)

## 6. Class Balance (recommended flag)

In [ ]:
balance = recs_sample['is_recommended'].value_counts(normalize=True).mul(100).round(1)
print(balance)
balance.plot(kind='bar', color=['#2ca02c', '#d62728'])
plt.title('Review Recommendation Balance (%)')
plt.xlabel('is_recommended')
plt.ylabel('%')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(FIG / 'prof02_class_balance.png', dpi=150)
plt.show()

**Note**: The dataset is heavily imbalanced — positive (recommended) reviews substantially outnumber negative ones. This must be accounted for in the P17 classifier (class weighting or under-sampling).